# 04 — Retained-data syndrome rounds

A **shot** encodes the three data qubits once, then repeats parity extraction on the *same data* for $R$ rounds. Each round measures two ancillas into its own classical register; the ancillas are reset between rounds. The data is read out at the end. Every CNOT has a symbolic target-$R_x(\theta_g)$ residual; the four syndrome CNOT parameters recur in each round, while encoding CNOTs occur once.

This notebook establishes the circuit and history format from scratch. It does not use earlier exploratory two-gate runs. No physical $X$ injection or active recovery is performed. Notebook 05 performs the new optimization sweep.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from qec_calibration import build_repeated_round_template, run_repeated_rounds, round_syndrome_loss

R = 3
model = build_repeated_round_template(3, rounds=R)
display(pd.DataFrame([{"index": i, "gate_id": loc.gate_id, "stage": loc.stage,
                      "control": loc.control, "target": loc.target} for i, loc in enumerate(model.locations)]))
print(model.circuit.draw(output="text", fold=100))


## One shot, several records

The return key is `(round_0, round_1, ..., round_(R-1), final_data_word)`. Syndrome bits are in conceptual stabilizer order $s_0s_1$, where $s_0$ is $Z_0Z_1$ and $s_1$ is $Z_1Z_2$. A final data word uses Qiskit's displayed register order. The data measurement is diagnostic only: it never enters the training loss.

In [ ]:
ideal = run_repeated_rounds(model, shots=128, seed=40)
assert ideal == {("00",) * R + ("000",): 128}
display(pd.DataFrame([{"syndromes": key[:-1], "final_data": key[-1], "shots": count}
                      for key, count in ideal.items()]))
print("Average per-round syndrome loss:", round_syndrome_loss(ideal, model))


## Controlled circuit checks

A fault on encoding target `data[1]` can leave a data $X_1$ component and repeated `11` records. A residual on an ancilla CNOT can produce intermittent measurement-bit errors. These checks verify the meaning of the stored histories; **neither history pattern is a training target**. Two target rotations on the same ancilla can cancel, which remains a source of parameter degeneracy even with repeated rounds.

In [ ]:
for gate in ("enc_0_1", "syn_0_left"):
    counts = run_repeated_rounds(model, {gate: 0.7}, shots=1024, seed=41)
    rows = [{"syndromes": " / ".join(key[:-1]), "final_data": key[-1],
             "shots": count} for key, count in counts.items()]
    print(f"{gate}: average per-round loss = {round_syndrome_loss(counts, model):.4f}")
    display(pd.DataFrame(rows).sort_values("shots", ascending=False).head(8))


## Training objective used next

For $N$ shots, define $s_j^{(t)}$ as the syndrome in round $t$ of shot $j$. Notebook 05 minimizes only

$$\widehat L_R(\boldsymbol\theta)=\frac{1}{NR}\sum_{j=1}^N\sum_{t=0}^{R-1}\mathbf 1[s_j^{(t)}\ne 00].$$

Full histories and final data are retained for separate descriptive analysis. The circuit includes mid-circuit measurement and reset, so retained **data** evolves across rounds, but the whole shot is not coherent unitary evolution.